In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bikestore.logistics;
CREATE EXTERNAL VOLUME IF NOT EXISTS bikestore.logistics.bikestore_resource COMMENT 'Volume projeto bikestore' LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource'



In [0]:
display(dbutils.fs.ls("/Volumes/bikestore/logistics/bikestore_resource/origem/"))
# Mapeamento volume oriegm

path,name,size,modificationTime
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/brands.csv,brands.csv,120,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/categories.csv,categories.csv,162,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/customers.csv,customers.csv,127671,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/desktop.ini,desktop.ini,106,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/order_items.csv,order_items.csv,112937,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/orders.csv,orders.csv,77286,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/products.csv,products.csv,17390,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/schema.png,schema.png,12013,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/staffs.csv,staffs.csv,726,1790254847000
dbfs:/Volumes/bikestore/logistics/bikestore_resource/origem/stocks.csv,stocks.csv,8773,1790254847000


In [0]:

display(dbutils.fs.ls("abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore"))
# Mapeamento externo

path,name,size,modificationTime
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/,bronze/,0,0
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/,gold/,0,0
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/,resource/,0,0
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/,silver/,0,0


In [0]:
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
display(dbutils.fs.ls(resource_path)) # Map variável

path,name,size,modificationTime
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/brands.csv,brands.csv,120,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/categories.csv,categories.csv,162,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/customers.csv,customers.csv,127671,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/desktop.ini,desktop.ini,106,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/order_items.csv,order_items.csv,112937,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/orders.csv,orders.csv,77286,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/products.csv,products.csv,17390,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/schema.png,schema.png,12013,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/staffs.csv,staffs.csv,726,1790254847000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem/stocks.csv,stocks.csv,8773,1790254847000


In [0]:
df_brands = spark.read.csv(f'{resource_path}/brands.csv',
                            header=True,
                            inferSchema=True,
                            sep=','                
                            )

In [0]:
display(df_brands)

brand_id,brand_name
1,Electra
2,Haro
3,Heller
4,Pure Cycles
5,Ritchey
6,Strider
7,Sun Bicycles
8,Surly
9,Trek


In [0]:
# Salvar em parquet como delta
df_brands.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema','true') \
    .save(f"{bronze_path}/brands")

In [0]:
display(dbutils.fs.ls(bronze_path))

path,name,size,modificationTime
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/bonze/,bonze/,0,1790269723000
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/brands/,brands/,0,1790269775000


In [0]:
%sql
-- Crie uma Tabela com o External Location 'não iremos usar na bronze' pois são dados brutos e precisam de tratamento (geralmente não são criados para esta parte, mas não é uma regra)
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_brands
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/brands/'


In [0]:
%sql
SELECT * FROM bikestore.logistics.bronze_brands

brand_id,brand_name
1,Electra
2,Haro
3,Heller
4,Pure Cycles
5,Ritchey
6,Strider
7,Sun Bicycles
8,Surly
9,Trek
